# 09 Feature Engineering

## 1. Objective

Objective:
Turn the EDA findings into engineered candidate features that can support later feature selection and modeling.

This notebook focuses on:
- defining feature logic from the earlier analysis
- creating transformed, grouped, interaction, and business features
- documenting redundancy and exploratory feature usefulness
- saving the engineered candidate dataset for the next stage


## Imports


In [85]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.feature_selection import chi2, mutual_info_classif

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CATEGORICAL_COLUMNS, NUMERICAL_COLUMNS, TARGET_COLUMN
from src.data.data_loader import load_cleaned_data


## 2. Load Data

Load the cleaned dataset and separate the target, numerical, and categorical feature groups.


In [86]:
df = load_cleaned_data()

target_column = TARGET_COLUMN
numerical_columns = [column for column in NUMERICAL_COLUMNS if column in df.columns]
categorical_columns = [column for column in CATEGORICAL_COLUMNS if column in df.columns]

feature_groups = {
    "target_column": target_column,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "shape": df.shape,
}

feature_groups


{'target_column': 'Churn',
 'numerical_columns': ['tenure', 'MonthlyCharges', 'TotalCharges'],
 'categorical_columns': ['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod'],
 'shape': (7043, 21)}

## 3. Revisit EDA Findings

- `tenure`, `Contract`, `PaymentMethod`, `TechSupport`, `OnlineSecurity`, and `InternetService` were among the strongest churn-related signals in the earlier notebooks.
- `tenure` and `TotalCharges` were strongly related, so lifecycle redundancy needs to be documented carefully.
- High-risk combinations such as `Month-to-month + Electronic check` and `Fiber optic + No TechSupport` suggest interaction and business features are worth creating.
- This notebook therefore focuses on feature design and preview, not on fitting the final leakage-safe preprocessing pipeline.


## 4. Numerical Feature Engineering

Create numerical features that reflect the earlier findings without fitting dataset-wide preprocessing objects. At this stage, the focus is on deterministic transformations such as log features and business-friendly bins.


In [87]:
feature_df = df.copy()

if "TotalCharges" in feature_df.columns:
    # Apply a log transform to reduce right skew while remaining safe for zero values.
    feature_df["log_total_charges"] = np.log1p(feature_df["TotalCharges"].fillna(0))
    # Drop the raw cumulative-charge feature to keep one clear representation and reduce redundancy.
    feature_df = feature_df.drop(columns=["TotalCharges"])

if "tenure" in feature_df.columns:
    feature_df["tenure_band"] = pd.cut(
        feature_df["tenure"],
        bins=[-0.1, 12, 24, 48, 72],
        labels=["new", "early", "mid", "long_term"],
    )

numerical_feature_summary = {
    "numerical_features_added": [
        column for column in ["log_total_charges", "tenure_band"] if column in feature_df.columns
    ],
    "feature_shape": feature_df.shape,
}

numerical_feature_summary


{'numerical_features_added': ['log_total_charges', 'tenure_band'],
 'feature_shape': (7043, 22)}

## 5. Categorical Feature Transformations

Transform categorical features in a lightweight way by grouping and binary mapping where the business meaning is clear. Full one-hot encoding is intentionally deferred to the later modeling pipeline.


In [88]:
# Map binary categories directly onto the original columns so each feature has one representation.
binary_feature_mappings = {
    "gender": {"Female": 0, "Male": 1},
    "Partner": {"No": 0, "Yes": 1},
    "Dependents": {"No": 0, "Yes": 1},
    "PhoneService": {"No": 0, "Yes": 1},
    "PaperlessBilling": {"No": 0, "Yes": 1},
}

for column, mapping in binary_feature_mappings.items():
    if column in feature_df.columns:
        feature_df[column] = feature_df[column].map(mapping).astype("Int64")

# SeniorCitizen is already binary in this dataset, so keep the original column and standardize its dtype.
if "SeniorCitizen" in feature_df.columns:
    feature_df["SeniorCitizen"] = feature_df["SeniorCitizen"].astype("Int64")

# Convert the label to a numeric target column and drop the original string label to avoid duplicate targets.
if target_column in feature_df.columns:
    feature_df[target_column] = feature_df[target_column].map({"No": 0, "Yes": 1}).astype("Int64")
    feature_df = feature_df.rename(columns={target_column: "target"})
    target_column = "target"

# Keep ordinal encoding only where order is meaningful.
if "Contract" in feature_df.columns:
    feature_df["Contract_ordinal"] = feature_df["Contract"].map(
        {"Month-to-month": 0, "One year": 1, "Two year": 2}
    ).astype("Int64")

# Keep grouped categorical features as categories for later one-hot encoding inside an sklearn pipeline.
if "PaymentMethod" in feature_df.columns:
    feature_df["payment_method_group"] = feature_df["PaymentMethod"].replace(
        {
            "Bank transfer (automatic)": "auto_payment",
            "Credit card (automatic)": "auto_payment",
            "Electronic check": "electronic_check",
            "Mailed check": "manual_check",
        }
    )

if "InternetService" in feature_df.columns:
    feature_df["internet_service_group"] = feature_df["InternetService"].replace(
        {
            "Fiber optic": "fiber",
            "DSL": "dsl",
            "No": "no_internet",
        }
    )

categorical_transform_summary = {
    "binary_encoded_in_place": [
        column for column in [*binary_feature_mappings, "SeniorCitizen", target_column] if column in feature_df.columns
    ],
    "grouped_features": [
        column for column in ["payment_method_group", "internet_service_group"] if column in feature_df.columns
    ],
    "ordinal_features": [column for column in ["Contract_ordinal"] if column in feature_df.columns],
    "one_hot_encode_later": [
        column
        for column in [
            "Contract",
            "PaymentMethod",
            "InternetService",
            "tenure_band",
            "payment_method_group",
            "internet_service_group",
            "contract_payment_profile",
            "internet_techsupport_profile",
            "internet_onlinesecurity_profile",
        ]
        if column in feature_df.columns
    ],
}

categorical_transform_summary


{'binary_encoded_in_place': ['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'PaperlessBilling',
  'SeniorCitizen',
  'target'],
 'grouped_features': ['payment_method_group', 'internet_service_group'],
 'ordinal_features': ['Contract_ordinal'],
 'one_hot_encode_later': ['Contract',
  'PaymentMethod',
  'InternetService',
  'tenure_band',
  'payment_method_group',
  'internet_service_group']}

## 6. Interaction Features

Create interaction-style features that reflect the multivariate risk combinations found earlier, using readable combined categories instead of full encoded cross-products.


In [89]:
if {"tenure", "MonthlyCharges"}.issubset(feature_df.columns):
    feature_df["tenure_x_MonthlyCharges"] = feature_df["tenure"] * feature_df["MonthlyCharges"]

if {"Contract", "PaymentMethod"}.issubset(feature_df.columns):
    feature_df["contract_payment_profile"] = (
        feature_df["Contract"].astype(str) + "__" + feature_df["PaymentMethod"].astype(str)
    )

if {"InternetService", "TechSupport"}.issubset(feature_df.columns):
    feature_df["internet_techsupport_profile"] = (
        feature_df["InternetService"].astype(str) + "__" + feature_df["TechSupport"].astype(str)
    )

if {"InternetService", "OnlineSecurity"}.issubset(feature_df.columns):
    feature_df["internet_onlinesecurity_profile"] = (
        feature_df["InternetService"].astype(str) + "__" + feature_df["OnlineSecurity"].astype(str)
    )

interaction_feature_summary = {
    "interaction_features": [
        column
        for column in [
            "tenure_x_MonthlyCharges",
            "contract_payment_profile",
            "internet_techsupport_profile",
            "internet_onlinesecurity_profile",
        ]
        if column in feature_df.columns
    ]
}

interaction_feature_summary


{'interaction_features': ['tenure_x_MonthlyCharges',
  'contract_payment_profile',
  'internet_techsupport_profile',
  'internet_onlinesecurity_profile']}

## 7. Business Features

Create business-friendly indicators that summarize customer state more directly than the raw columns alone.


In [90]:
if {"log_total_charges", "tenure"}.issubset(feature_df.columns):
    # Recover the original scale only inside this derived metric so we do not keep TotalCharges itself.
    total_charges_from_log = np.expm1(feature_df["log_total_charges"])
    tenure_nonzero = feature_df["tenure"].replace(0, np.nan)
    feature_df["avg_monthly_spend"] = (
        total_charges_from_log / tenure_nonzero
    ).fillna(feature_df["MonthlyCharges"])

if "tenure" in feature_df.columns:
    feature_df["is_new_customer"] = (feature_df["tenure"] < 12).astype(int)

support_source_columns = [
    column for column in ["TechSupport", "OnlineSecurity", "OnlineBackup"] if column in feature_df.columns
]
if support_source_columns:
    feature_df["has_support_services"] = (
        feature_df[support_source_columns].eq("Yes").any(axis=1)
    ).astype(int)

if "Contract" in feature_df.columns:
    feature_df["is_month_to_month"] = (feature_df["Contract"] == "Month-to-month").astype(int)

if "PaymentMethod" in feature_df.columns:
    feature_df["is_auto_payment"] = feature_df["PaymentMethod"].isin(
        ["Bank transfer (automatic)", "Credit card (automatic)"]
    ).astype(int)

service_count_columns = [
    column
    for column in [
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]
    if column in feature_df.columns
]
if service_count_columns:
    feature_df["service_count"] = feature_df[service_count_columns].eq("Yes").sum(axis=1)

business_feature_summary = {
    "business_features": [
        column
        for column in [
            "avg_monthly_spend",
            "is_new_customer",
            "has_support_services",
            "is_month_to_month",
            "is_auto_payment",
            "service_count",
        ]
        if column in feature_df.columns
    ]
}

business_feature_summary


{'business_features': ['avg_monthly_spend',
  'is_new_customer',
  'has_support_services',
  'is_month_to_month',
  'is_auto_payment',
  'service_count']}

## 8. Redundancy Handling

The raw `TotalCharges` feature is log-transformed into `log_total_charges` to reduce right skew and then removed so the dataset keeps only one cumulative-charge representation. This makes the transformation explicit, avoids carrying both raw and transformed versions into feature selection, and reduces redundancy with other spend-related variables. `avg_monthly_spend` is still retained as a separate business feature because it captures spending intensity rather than cumulative value.

`Contract`, `Contract_ordinal`, and `is_month_to_month` are co-derived representations of the same original variable. They should not all enter the final model together. A single representation should be chosen later depending on model type, with `Contract_ordinal` generally being more suitable for tree-based models and `is_month_to_month` being a simpler option for linear models when the main risk boundary is month-to-month contracts.


## 9. Feature Selection Support (light)

Use simple exploratory ranking methods to see which engineered features look promising. This is only support analysis and should not be treated as the final leakage-safe selection step. Pearson-style correlation and mutual information are used for continuous features, while chi-square is restricted to binary, grouped categorical, and count-like engineered features.


In [91]:
selection_df = feature_df.copy()

# Convert remaining categorical predictors to integer codes only for notebook-side filter methods.
# The production sklearn pipeline should still handle one-hot encoding after the train/test split.
for column in selection_df.select_dtypes(include=["object", "category"]).columns:
    if column != target_column:
        selection_df[column] = selection_df[column].astype("category").cat.codes

y = selection_df[target_column].astype(int)

drop_columns = [column for column in [target_column, "customerID"] if column in selection_df.columns]
X = selection_df.drop(columns=drop_columns)
assert target_column not in X.columns, "Target leaked into X"

correlation_rows = []
for column in X.columns:
    correlation_value = pd.Series(X[column]).corr(y)
    correlation_rows.append(
        {
            "feature": column,
            "target_correlation": round(float(correlation_value), 4),
            "abs_target_correlation": round(abs(float(correlation_value)), 4),
        }
    )

correlation_rank_df = pd.DataFrame(correlation_rows).sort_values(
    "abs_target_correlation", ascending=False
)

chi2_candidate_columns = [
    column for column in [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "PaperlessBilling",
        "SeniorCitizen",
        "Contract_ordinal",
        "payment_method_group",
        "internet_service_group",
        "contract_payment_profile",
        "internet_techsupport_profile",
        "internet_onlinesecurity_profile",
        "is_new_customer",
        "has_support_services",
        "is_month_to_month",
        "is_auto_payment",
        "service_count",
    ]
    if column in X.columns
]

chi2_input = X[chi2_candidate_columns].copy()
chi2_scores, chi2_p_values = chi2(chi2_input.clip(lower=0), y)
chi2_rank_df = pd.DataFrame(
    {
        "feature": chi2_input.columns,
        "chi2_score": np.round(chi2_scores, 4),
        "chi2_p_value": chi2_p_values,
    }
).sort_values("chi2_score", ascending=False)

mutual_info_scores = mutual_info_classif(X, y, random_state=42)
mutual_info_rank_df = pd.DataFrame(
    {
        "feature": X.columns,
        "mutual_information": np.round(mutual_info_scores, 4),
    }
).sort_values("mutual_information", ascending=False)

feature_selection_support_df = (
    correlation_rank_df.merge(chi2_rank_df, on="feature", how="outer")
    .merge(mutual_info_rank_df, on="feature", how="outer")
    .sort_values(["mutual_information", "abs_target_correlation"], ascending=[False, False])
)

display(feature_selection_support_df.head(20))


,feature,target_correlation,abs_target_correlation,chi2_score,chi2_p_value,mutual_information
18,contract_payment_profile,-0.3705,0.3705,2372.5516,0.000000e+00,0.1180
1,Contract_ordinal,-0.3967,0.3967,1115.7802,1.227941e-244,0.1066
0,Contract,-0.3967,0.3967,NaN,NaN,0.1008
25,is_month_to_month,0.4051,0.4051,519.8953,4.459832e-115,0.0909
30,tenure,-0.3522,0.3522,NaN,NaN,0.0836
21,internet_onlinesecurity_profile,-0.1080,0.1080,75.0900,4.497274e-18,0.0822
8,OnlineSecurity,-0.2893,0.2893,NaN,NaN,0.0751
23,internet_techsupport_profile,-0.1059,0.1059,72.2444,1.901282e-17,0.0726
31,tenure_band,-0.3454,0.3454,NaN,NaN,0.0638
16,TechSupport,-0.2825,0.2825,NaN,NaN,0.0572


The feature-selection support table suggests that contract-related signals remain the strongest engineered predictors, with `contract_payment_profile`, `Contract_ordinal`, and `is_month_to_month` all ranking near the top. This confirms that contract structure is central to churn risk, but it also highlights an important modeling caveat: grouped profile features such as `contract_payment_profile`, `internet_techsupport_profile`, and `internet_onlinesecurity_profile` are informative combined categories that will require careful encoding later. For cleaner baseline models, simpler features such as `Contract_ordinal`, `is_month_to_month`, `tenure`, `tenure_band`, and `is_new_customer` are easier to carry forward and interpret.


## 10. Save Feature-Engineered Dataset

Save the engineered candidate dataset and feature list as the output artifact of feature engineering. This is a handoff dataset for later stages, not the final train/test-ready modeling matrix.


In [92]:
feature_engineering_tables_dir = project_root / "reports" / "tables" / "feature_engineering"
feature_engineering_tables_dir.mkdir(parents=True, exist_ok=True)

processed_data_dir = project_root / "data" / "processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)

final_engineered_df = feature_df.copy()
feature_columns = [column for column in final_engineered_df.columns if column != target_column]

report_dataset_path = feature_engineering_tables_dir / "feature_engineered_dataset.csv"
processed_dataset_path = processed_data_dir / "telco_churn_engineered.csv"
feature_list_path = feature_engineering_tables_dir / "feature_list.csv"

final_engineered_df.to_csv(report_dataset_path, index=False)
final_engineered_df.to_csv(processed_dataset_path, index=False)
pd.Series(feature_columns, name="feature").to_csv(feature_list_path, index=False)

final_dataset_summary = {
    "dataset_shape": final_engineered_df.shape,
    "feature_count": len(feature_columns),
    "target_column": target_column,
    "report_dataset_path": str(report_dataset_path.relative_to(project_root)),
    "processed_dataset_path": str(processed_dataset_path.relative_to(project_root)),
    "feature_list_path": str(feature_list_path.relative_to(project_root)),
}

final_dataset_summary


{'dataset_shape': (7043, 35),
 'feature_count': 34,
 'target_column': 'target',
 'report_dataset_path': 'reports/tables/feature_engineering/feature_engineered_dataset.csv',
 'processed_dataset_path': 'data/processed/telco_churn_engineered.csv',
 'feature_list_path': 'reports/tables/feature_engineering/feature_list.csv'}

## 11. Summary

This notebook now keeps binary features in a single, clean representation by encoding the original columns directly and converting the churn label into a numeric `target` column. It also replaces raw `TotalCharges` with a single `log_total_charges` feature so skew is handled without keeping redundant versions of the same signal. Higher-cardinality or grouped categorical variables such as `Contract`, `PaymentMethod`, `InternetService`, `tenure_band`, and the profile features are intentionally left as categorical signals so they can be one-hot encoded later inside a leakage-safe modeling pipeline.
